Import necessary packages:

In [1]:
from collections import deque
import itertools
from sage.all import *
from pysat.formula import *
from pysat.solvers import *

# Compute multiplication table for $B(n)$:

We use the matrix representation of $P$ from the Gardam paper (https://arxiv.org/abs/2312.05240), which makes solving the word problem in $P$ reduce to matrix multiplication over integer matrices.

In [2]:
K = GF(2)
gens = [matrix(QQ, 4, [1, 0, 0, 1, 0, -1, 0, 1, 0, 0, -1, 0, 0, 0, 0, 1]), matrix(QQ, 4, [-1, 0, 0, 0, 0, 1, 0, 1, 0, 0, -1, 1, 0, 0, 0, 1])]
P = MatrixGroup(gens)

symbols = [P.one(), P.gen(0), P.gen(0).inverse(), P.gen(1), P.gen(1).inverse()]
B_5 = [prod(word) for word in itertools.product(symbols, repeat=5)]
B_5 = list(set(B_5))
print(len(B_5))
print(B_5[0])

147
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]


In [3]:
DEBUG = False

if DEBUG:
    K = GF(2)
    reset('i')
    # gens = [matrix(GF(11), 2, [2, 0, 0, 2]),]
    gens = [matrix(ZZ, 2, [0, -1, 1, 0])]
    P = MatrixGroup(gens)
    
    symbols = [P.one(), P.gen(0), P.gen(0).inverse()]
    B_5 = [prod(word) for word in itertools.product(symbols, repeat=4)]
    B_5 = list(set(B_5))
    print(len(B_5))
    print(B_5[0])
    print(B_5) # For small G

For $G = Z_4$, the SAT solver finds the solution $\alpha = \beta = 0 + 1 + 3.$

In [4]:
# Keys are the product, values are lists of pairs realizing the product
product_table = dict()

factors_to_product = dict() # keys are pairs of indices, values are unique ids for the product. used for the gym environment.
unique_id = 0
for i,j in itertools.product(range(len(B_5)), repeat=2):
    a, b = B_5[i], B_5[j]
    val = a*b
    if val in product_table:
        product_table[val].append((i, j))
        factors_to_product[(i,j)] = factors_to_product[product_table[val][0]]
    else:
        product_table[val] = [(i,j),]
        factors_to_product[(i,j)] = unique_id
        unique_id += 1
print(len(product_table))

981


# Reinforcement Learning Routine

In [11]:
import gymnasium as gym
import numpy as np
from gymnasium.spaces import Discrete, MultiBinary

N = len(B_5)
p = 2
REWARD_THRESHOLD = 0
MAX_EPISODE_STEPS = 1000

class ZeroDivisorEnv(gym.Env):
    def __init__(self):
        self.observation_space = MultiBinary(2*N)
        self.action_space = Discrete(2*N)
        self.alpha = np.zeros(N, dtype=np.int8)
        self.beta = np.zeros(N, dtype=np.int8)

    def get_obs(self):
        return np.concatenate((self.alpha, self.beta))

    def get_info(self):
        return {"support_alpha": np.count_nonzero(self.alpha), "support_beta": np.count_nonzero(self.beta)}

    def reward(self):
        return -1 * self.product_support() / (np.count_nonzero(self.alpha) * np.count_nonzero(self.beta))

    def product_support(self):
        product_dict = dict()
        for i in range(N):
            for j in range(N):
                if self.alpha[i] and self.beta[j]:
                    val = factors_to_product[(i,j)]
                    if val in product_dict:
                        product_dict[val] += 1
                    else:
                        product_dict[val] = 1

        support = 0
        for val in product_dict:
            support += product_dict[val] % p
        return support
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        while np.count_nonzero(self.alpha) <= 0:
            self.alpha = np.random.randint(2, size=N, dtype=np.int8)
        while np.count_nonzero(self.beta) <= 0:
            self.beta = np.random.randint(2, size=N, dtype=np.int8)
        
        observation = self.get_obs()
        info = self.get_info()

        return observation, info

    def step(self, action):
        if action < N:
            self.alpha[action] = (self.alpha[action] + 1) % 2
        elif action >= N:
            self.alpha[action % N] = (self.beta[action % N] + 1) % 2

        terminated = True if self.product_support() == 0 else False
        truncated = True if np.count_nonzero(self.alpha) == 0 or np.count_nonzero(self.beta) == 0 else False
        reward = self.reward()
        observation = self.get_obs()
        info = self.get_info()

        return observation, reward, terminated, truncated, info

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

# Parallel environments
vec_env = make_vec_env(ZeroDivisorEnv, n_envs=4, vec_env_cls=SubprocVecEnv)

model = PPO("MlpPolicy", vec_env, verbose=1, device="cpu")
model.learn(total_timesteps=25000)
model.save("ppo_zerodivisor")

obs = vec_env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = vec_env.step(action)

Using cpu device


# Assert System of Boolean Equations:

In [5]:
a_vars = [Atom(f"a_{i}") for i in range(len(B_5))]
b_vars = [Atom(f"b_{j}") for j in range(len(B_5))]

x_vars = dict()
cnf = CNF()

# Non triviality
formula = Equals(a_vars[0], PYSAT_TRUE)
cnf.extend([c for c in formula])

formula = Or(*list(a_vars[i] for i in range(1, len(B_5))))
cnf.extend([c for c in formula])

# Product equations x_g,h = a_g * b_h
for i,j in itertools.product(range(len(B_5)), repeat=2):
    x_vars[(i, j)] = Atom(f"x_{i}{j}")
    formula = Equals(x_vars[(i,j)], And(a_vars[i], b_vars[j]))
    cnf.extend([c for c in formula])

max_id = -1
for c in cnf.clauses:
    for variable in c:
        if abs(variable) > max_id:
            max_id = abs(variable)

# print(max_id)
next_id = max_id + 1


# Sum equations sum_{gh=k}(x_g,h) = delta(1,k) for each k in the product table
var_list = deque([x_vars[(i,j)] for i,j in product_table[P.one()]])
formula_list = []
while var_list:
    if len(var_list) == 1:
        formula = Equals(var_list.pop(), PYSAT_TRUE)
        formula_list.append(formula)
    elif len(var_list) > 2:
        x1 = var_list.pop()
        x2 = var_list.pop()
        aux_var = Atom(next_id)
        var_list.append(aux_var)
        next_id += 1
        formula = Equals(XOr(x1, x2), aux_var)
        formula_list.append(formula)
    elif len(var_list) == 2:
        x1 = var_list.pop()
        x2 = var_list.pop()
        formula = XOr(x1, x2)
        formula_list.append(formula)
for f in formula_list:
    cnf.extend([c for c in f])
        


"""
if len(product_table[P.one()]) > 1:
    formula = XOr(*[x_vars[(i,j)] for i,j in product_table[P.one()]])
    cnf.extend([c for c in formula])
else:
    formula = Equals(x_vars[product_table[P.one()][0]], PYSAT_TRUE)
    cnf.extend([c for c in formula])
"""
for c in cnf.clauses:
    for variable in c:
        if abs(variable) > max_id:
            max_id = abs(variable)
next_id = max_id + 1

formula_list = []
for val in product_table:
    if not val.is_one():
        
        var_list = deque([x_vars[(i,j)] for i,j in product_table[val]])
        while len(var_list) > 0:
            if len(var_list) == 1:
                formula = Neg(var_list.pop())
                formula_list.append(formula)
            elif len(var_list) > 2:
                x1 = var_list.pop()
                x2 = var_list.pop()
                aux_var = Atom(next_id)
                var_list.append(aux_var)
                next_id += 1
                formula = Equals(XOr(x1, x2), aux_var)
                formula_list.append(formula)
            elif len(var_list) == 2:
                x1 = var_list.pop()
                x2 = var_list.pop()
                formula = Neg(XOr(x1, x2))
                formula_list.append(formula)
        
        """
        if len(product_table[val]) > 1:
            formula = Neg(XOr(*[x_vars[(i,j)] for i,j in product_table[val]]))
            cnf.extend([c for c in formula])
        else:
            formula = Neg(x_vars[product_table[val][0]])
            cnf.extend([c for c in formula])
        """
for f in formula_list:
    cnf.extend([c for c in f])

for c in cnf.clauses:
    for variable in c:
        if abs(variable) > max_id:
            max_id = abs(variable)

print(max_id)

82040


In [ ]:
solution = []
with Minisat22(bootstrap_with=cnf.clauses) as m:
    print(m.solve())
    solution = m.get_model()
support = list(filter(lambda n: n > 0, solution))
print(support)
# print(solution)

In [ ]:
obj2id = Formula.export_vpool().obj2id

for i in range(len(B_5)):
    if obj2id[a_vars[i]] in support:
        print(f"a_{i}")
    if obj2id[b_vars[i]] in support:
        print(f"b_{i}")

for i,j in itertools.product(range(len(B_5)), repeat=2):
    if obj2id[x_vars[(i,j)]] in support:
        print(f"x_{i},{j}")

# Solve as Multivariate Polynomial System

Uses Singular's `triangular_decomposition`.

In [ ]:
from sage.rings.polynomial.msolve import variety

a_var_names = tuple(f"a_{i}" for i in range(len(B_5)))
b_var_names = tuple(f"b_{j}" for j in range(len(B_5)))

R = BooleanPolynomialRing(names=a_var_names+b_var_names)
a_vars = R.gens()[:len(B_5)]
b_vars = R.gens()[len(B_5):]

eqns = [sum(a_vars[i]*b_vars[j] for i,j in product_table[P.one()]), a_vars[0] - 1, b_vars[0] - 1]
print("calculating sum eqns...")
for val in product_table:
    if val.is_one():
        continue
    eqns.append(sum(a_vars[i]*b_vars[j] for i,j in product_table[val]))
I = Ideal(eqns)
gb = I.groebner_basis(algorithm='msolve', proof=False)
print(list(gb))
print(I.dimension())
# sorted(variety(I, R, proof=False), key=str)